# Rectified Flow with Different Schedulers (PyTorch)

In the [previous tutorial](./rectified_flow_tutorial.ipynb) we trained
a rectified flow using the **linear** interpolation
$\mathbf{x}_t = (1-t)\,\mathbf{x}_0 + t\,\mathbf{x}_1$. That was a
specific choice. The same rectified-flow algorithm works for any
smooth interpolation

$$\mathbf{x}_t \;=\; \alpha_t\, \mathbf{x}_0 \;+\; \sigma_t\, \mathbf{x}_1,
\qquad \mathbf{x}_0 \sim p_{\text{src}},\;\; \mathbf{x}_1 \sim p_{\text{tgt}},$$

and different choices of $(\alpha_t, \sigma_t)$ correspond to
different popular schedulers from the diffusion / flow-matching
literature. In this notebook we'll implement and compare two:

| Scheduler | $\alpha_t$ | $\sigma_t$ | Origin |
|-----------|------------|------------|--------|
| **Linear** (Flow Matching) | $1-t$ | $t$ | Lipman et al. 2022; Liu et al. 2022 |
| **VP-Trig** (DDPM cosine) | $\cos(\pi t/2)$ | $\sin(\pi t/2)$ | Nichol & Dhariwal 2021 |

Both schedulers satisfy the boundary conditions $\alpha_0 = 1,
\sigma_0 = 0$ (so $\mathbf{x}_t = \mathbf{x}_0$ at $t=0$) and
$\alpha_1 = 0, \sigma_1 = 1$ (so $\mathbf{x}_t = \mathbf{x}_1$ at
$t=1$). They both transport $p_{\text{src}}$ to $p_{\text{tgt}}$
correctly — but along **different paths**, as we'll see.

**What you'll learn.**

1. The generalized rectified-flow loss for any $(\alpha_t, \sigma_t)$.
2. Why **Linear** traces straight segments while **VP-Trig** traces
   *curved arcs* — and why both still produce correct transport.
3. How **Reflow** behaves under each scheduler.

**Implementation.** Pure PyTorch. Runs in ~3 minutes on a free
Colab CPU runtime; faster on GPU.


## 1. Setup


In [ ]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams["font.family"] = "serif"
rcParams["font.size"]   = 11

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Colors
C_SRC = "#7C3AED"   # purple — source samples
C_TGT = "#DC2626"   # red    — target samples
C_LIN = "#2563EB"   # blue   — Linear / FM
C_VP  = "#10B981"   # green  — VP-Trig
C_BLUE  = "#2563EB"
C_GREEN = "#10B981"

print(f"PyTorch {torch.__version__} on {device}")


## 2. The two distributions

Same 8-mode source/target rings as the previous tutorial.
- $p_{\text{src}}$: 8-mode Gaussian mixture on the **outer ring**
  (radius 4, angles offset by half a cluster).
- $p_{\text{tgt}}$: 8-mode Gaussian mixture on the **inner ring**
  (radius 1.1).

The half-cluster offset is what creates coupling ambiguity — each
source mode is equidistant from two target modes, so the network
is forced to learn a non-trivial averaged velocity field, which is
exactly what produces the curved trajectories we want to study.


In [ ]:
N_CLUSTERS  = 8
R_OUTER     = 4.0
R_INNER     = 1.1
SIGMA_OUTER = 0.16
SIGMA_INNER = 0.11

def sample_p_src(n, gen=None):
    """Source distribution: 8-mode mixture on the OUTER ring."""
    cid = torch.randint(0, N_CLUSTERS, (n,), generator=gen)
    a = cid.float() * (2 * math.pi / N_CLUSTERS) + math.pi / N_CLUSTERS
    centers = torch.stack([R_OUTER * torch.cos(a), R_OUTER * torch.sin(a)], -1)
    return centers + torch.randn(n, 2, generator=gen) * SIGMA_OUTER

def sample_p_tgt(n, gen=None):
    """Target distribution: 8-mode mixture on the INNER ring (no offset)."""
    cid = torch.randint(0, N_CLUSTERS, (n,), generator=gen)
    a = cid.float() * (2 * math.pi / N_CLUSTERS)
    centers = torch.stack([R_INNER * torch.cos(a), R_INNER * torch.sin(a)], -1)
    return centers + torch.randn(n, 2, generator=gen) * SIGMA_INNER

# Visualize
gen = torch.Generator().manual_seed(0)
src_vis = sample_p_src(600, gen).numpy()
tgt_vis = sample_p_tgt(600, gen).numpy()

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(src_vis[:, 0], src_vis[:, 1], s=10, c=C_SRC, alpha=0.7,
           label=r"source $p_{\text{src}}$")
ax.scatter(tgt_vis[:, 0], tgt_vis[:, 1], s=10, c=C_TGT, alpha=0.7,
           label=r"target $p_{\text{tgt}}$")
ax.set_aspect("equal"); ax.axis("off"); ax.legend(frameon=False)
ax.set_title("Source and target distributions")
plt.show()


## 3. The generalized rectified-flow framework

We define the interpolation between $\mathbf{x}_0\sim p_{\text{src}}$
and $\mathbf{x}_1\sim p_{\text{tgt}}$ in a unified way:

$$\mathbf{x}_t \;=\; \alpha_t\, \mathbf{x}_0 \;+\; \sigma_t\, \mathbf{x}_1.$$

Differentiating in $t$, the **conditional velocity** along the
interpolation path is

$$\dot{\mathbf{x}}_t \;=\; \dot\alpha_t\, \mathbf{x}_0 \;+\; \dot\sigma_t\, \mathbf{x}_1.$$

We learn the **marginal velocity field** $v_{\boldsymbol{\phi}}(\mathbf{x}, t)$
(a neural network with parameters $\boldsymbol{\phi}$) by least-squares
regression against the conditional velocity:

$$
\mathcal{L}(\boldsymbol{\phi}) \;=\;
\mathbb{E}_{\mathbf{x}_0,\,\mathbf{x}_1,\,t}\!
\left[\,\big\|\, v_{\boldsymbol{\phi}}(\mathbf{x}_t, t) \;-\; \dot\alpha_t\, \mathbf{x}_0 \;-\; \dot\sigma_t\, \mathbf{x}_1\,\big\|^2\,\right].
$$

When $\alpha_t = 1-t$ and $\sigma_t = t$ we have $\dot\alpha_t=-1$,
$\dot\sigma_t=1$, and the regression target is just $\mathbf{x}_1 - \mathbf{x}_0$ —
the original rectified-flow loss. Other schedulers give different
regression targets.


## 4. Two concrete schedulers

Each scheduler is a small Python class with four methods:
$\alpha_t$, $\sigma_t$, and their derivatives. Everything else
(training, sampling) is shared.


In [ ]:
class Scheduler:
    """Base class.  Subclasses implement alpha, sigma, and their derivatives."""
    name = "abstract"

    def alpha(self, t):     raise NotImplementedError
    def sigma(self, t):     raise NotImplementedError
    def alpha_dot(self, t): raise NotImplementedError
    def sigma_dot(self, t): raise NotImplementedError

    def x_t(self, x0, x1, t):
        """Conditional path: x_t = alpha_t x_0 + sigma_t x_1."""
        return self.alpha(t) * x0 + self.sigma(t) * x1

    def cond_velocity(self, x0, x1, t):
        """Conditional velocity along that path: d(x_t)/dt."""
        return self.alpha_dot(t) * x0 + self.sigma_dot(t) * x1


class LinearScheduler(Scheduler):
    """Flow matching / rectified flow.  Straight-line conditional path."""
    name = "Linear (FM)"

    def alpha(self, t):     return 1.0 - t
    def sigma(self, t):     return t
    def alpha_dot(self, t): return -torch.ones_like(t)
    def sigma_dot(self, t): return  torch.ones_like(t)


class VPTrigScheduler(Scheduler):
    """Variance-preserving trigonometric (DDPM cosine schedule).
    Variance preserved: alpha_t^2 + sigma_t^2 = 1.  Path is a curved arc."""
    name = "VP-Trig (DDPM)"

    def alpha(self, t):     return torch.cos(math.pi * t / 2)
    def sigma(self, t):     return torch.sin(math.pi * t / 2)
    def alpha_dot(self, t): return -(math.pi / 2) * torch.sin(math.pi * t / 2)
    def sigma_dot(self, t): return  (math.pi / 2) * torch.cos(math.pi * t / 2)


sched_lin = LinearScheduler()
sched_vp  = VPTrigScheduler()
schedulers = [sched_lin, sched_vp]
sched_colors = [C_LIN, C_VP]


## 5. Visualize the schedules

Plot $\alpha_t$ and $\sigma_t$ as functions of $t$.


In [ ]:
t_grid = torch.linspace(0, 1, 200).unsqueeze(-1)   # (200, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for sch, color in zip(schedulers, sched_colors):
    axes[0].plot(t_grid.numpy(), sch.alpha(t_grid).numpy(),
                 color=color, lw=2.2, label=sch.name)
    axes[1].plot(t_grid.numpy(), sch.sigma(t_grid).numpy(),
                 color=color, lw=2.2, label=sch.name)
axes[0].set_title(r"$\alpha_t$ (weight on $\mathbf{x}_0$)")
axes[1].set_title(r"$\sigma_t$ (weight on $\mathbf{x}_1$)")
for ax in axes:
    ax.set_xlabel(r"$t$"); ax.legend(frameon=False)
    ax.grid(alpha=0.25); ax.set_ylim(-0.05, 1.10)
plt.tight_layout(); plt.show()


**Read-out.**

* **Linear** has $\alpha_t + \sigma_t = (1-t) + t = 1$ for every
  $t$. That means $\mathbf{x}_t$ is a *convex combination* of
  $\mathbf{x}_0$ and $\mathbf{x}_1$, so the conditional path is a
  straight line segment.
* **VP-Trig** has the same endpoints, but
  $\alpha_t + \sigma_t = \cos(\pi t/2) + \sin(\pi t/2)$ peaks at
  $\sqrt{2} \approx 1.41$ at $t=1/2$. The coefficients no longer
  sum to 1, so the path is *not* a convex combination — it bows
  *outward* from the straight segment. What VP-Trig does preserve
  is the squared sum: $\alpha_t^2 + \sigma_t^2 = 1$ throughout.
  That's the "variance-preserving" property in the diffusion
  literature.


### Sample paths

Pick one $(\mathbf{x}_0, \mathbf{x}_1)$ pair per cluster (so the
eight pairs are spread symmetrically around the rings) and trace
the conditional path $\mathbf{x}_t$ for each scheduler:


In [ ]:
def one_per_cluster(seed=7):
    rng = np.random.default_rng(seed)
    pts0, pts1 = [], []
    for i in range(N_CLUSTERS):
        a0 = i * (2 * np.pi / N_CLUSTERS) + np.pi / N_CLUSTERS
        a1 = i * (2 * np.pi / N_CLUSTERS)
        pts0.append([R_OUTER * np.cos(a0), R_OUTER * np.sin(a0)]
                    + rng.standard_normal(2) * SIGMA_OUTER)
        pts1.append([R_INNER * np.cos(a1), R_INNER * np.sin(a1)]
                    + rng.standard_normal(2) * SIGMA_INNER)
    return np.array(pts0, dtype=np.float32), np.array(pts1, dtype=np.float32)

x0_demo, x1_demo = one_per_cluster()
t_path = torch.linspace(0, 1, 60).unsqueeze(-1)   # (60, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), facecolor="white")
for ax, sch, color in zip(axes, schedulers, sched_colors):
    a_t = sch.alpha(t_path).numpy()[:, None, :]    # (60, 1, 1)
    s_t = sch.sigma(t_path).numpy()[:, None, :]
    paths = a_t * x0_demo[None] + s_t * x1_demo[None]   # (60, 8, 2)
    for i in range(N_CLUSTERS):
        ax.plot(paths[:, i, 0], paths[:, i, 1], color=color, lw=1.6, alpha=0.85)
    ax.scatter(x0_demo[:, 0], x0_demo[:, 1], s=42, c=C_SRC,
               edgecolors="white", linewidths=0.7, zorder=10)
    ax.scatter(x1_demo[:, 0], x1_demo[:, 1], s=42, c=C_TGT,
               edgecolors="white", linewidths=0.7, zorder=10)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_xlim(-5.5, 5.5); ax.set_ylim(-5.5, 5.5)
    ax.set_title(sch.name, fontsize=14, color=color, pad=4)

fig.text(0.5, 0.04,
         r"purple = $\mathbf{x}_0$ (source)        red = $\mathbf{x}_1$ (target)",
         ha="center", fontsize=11)
plt.subplots_adjust(left=0.005, right=0.995, top=0.94, bottom=0.10, wspace=0.02)
plt.show()


Now you can *see* the geometric difference:

* **Linear** traces straight segments — the shortest path between
  each source/target pair.
* **VP-Trig** traces curved arcs that bulge outward, away from the
  origin. The variance-preserving constraint
  $\alpha_t^2 + \sigma_t^2 = 1$ keeps the magnitude of
  $\mathbf{x}_t$ from collapsing in the middle of the path.

This is the *conditional* path — what the path looks like when we
already know which pair $(\mathbf{x}_0, \mathbf{x}_1)$ we're
interpolating. The trained ODE will integrate the *marginal*
velocity, which is the conditional expectation
$\mathbb{E}[\dot{\mathbf{x}}_t \mid \mathbf{x}_t]$ — and that's
what produces additional curvature from the coupling ambiguity.


## 6. The velocity network $v_{\boldsymbol{\phi}}$

A small MLP with a Fourier embedding of $t$. The Fourier features
help the network represent rapidly-varying functions of $t$
without much capacity. ~38k parameters $\boldsymbol{\phi}$.


In [ ]:
class VelocityNet(nn.Module):
    """v_phi(x, t):  R^2 x [0,1] -> R^2."""
    def __init__(self, n_freqs=6, hidden=96, n_hidden_layers=4):
        super().__init__()
        # Log-spaced Fourier frequencies, registered as a non-trainable buffer
        freqs = (2.0 ** torch.arange(n_freqs)) * math.pi
        self.register_buffer("freqs", freqs)

        in_dim = 2 + 2 * n_freqs
        layers = [nn.Linear(in_dim, hidden), nn.SiLU()]
        for _ in range(n_hidden_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.SiLU()]
        layers += [nn.Linear(hidden, 2)]
        self.net = nn.Sequential(*layers)

    def fourier_features(self, t):
        # t: (B, 1) -> (B, 2*n_freqs)
        angles = t * self.freqs                       # broadcasts (B, n_freqs)
        return torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)

    def forward(self, x, t):
        h = torch.cat([x, self.fourier_features(t)], dim=-1)
        return self.net(h)

# Quick sanity check
demo_net = VelocityNet().to(device)
print(f"Network output shape: "
      f"{demo_net(torch.zeros(4, 2, device=device), torch.zeros(4, 1, device=device)).shape}")
print(f"Parameter count    : {sum(p.numel() for p in demo_net.parameters()):,}")


## 7. Training loop

The training loop is the standard PyTorch pattern: sample a batch,
compute the loss, backpropagate, step the optimizer. The scheduler
enters only through `scheduler.x_t` (which builds $\mathbf{x}_t$)
and `scheduler.cond_velocity` (which builds the regression target).


In [ ]:
def train_velocity(x0_data, x1_data, scheduler,
                   n_iters=3500, batch_size=512, lr=2e-3, label=""):
    """Train v_phi to match the conditional velocity of `scheduler`."""
    net = VelocityNet().to(device)
    optim = Adam(net.parameters(), lr=lr)
    lr_sched = CosineAnnealingLR(optim, T_max=n_iters, eta_min=lr * 0.05)
    n = x0_data.shape[0]

    t0 = time.time()
    for i in range(n_iters):
        idx = torch.randint(0, n, (batch_size,), device=device)
        x0_b = x0_data[idx]
        x1_b = x1_data[idx]
        t_b  = torch.rand(batch_size, 1, device=device)

        x_t    = scheduler.x_t(x0_b, x1_b, t_b)
        v_pred = net(x_t, t_b)
        v_true = scheduler.cond_velocity(x0_b, x1_b, t_b)
        loss   = ((v_pred - v_true) ** 2).sum(dim=-1).mean()

        optim.zero_grad()
        loss.backward()
        optim.step()
        lr_sched.step()

        if i % 1500 == 0 or i == n_iters - 1:
            print(f"  [{label}] iter {i:5d}  loss={loss.item():.4f}"
                  f"   t={time.time()-t0:5.1f}s")
    return net


## 8. ODE integration (sampling)

At sampling time we just integrate
$\dot{\mathbf{x}} = v_{\boldsymbol{\phi}}(\mathbf{x}, t)$ from
$t=0$ (where we sample $\mathbf{x} \sim p_{\text{src}}$) to $t=1$.
Forward Euler is enough for this 2D problem:

$$\mathbf{x}_{k+1} \;=\; \mathbf{x}_k \;+\; \tfrac{1}{N}\, v_{\boldsymbol{\phi}}\!\left(\mathbf{x}_k,\, \tfrac{k}{N}\right).$$


In [ ]:
@torch.no_grad()
def integrate(net, x0, n_steps=100, return_trajectory=False):
    net.eval()
    dt = 1.0 / n_steps
    x = x0.clone()
    traj = [x.detach().cpu().numpy().copy()] if return_trajectory else None
    for k in range(n_steps):
        t_b = torch.full((x.shape[0], 1), k / n_steps, device=x.device)
        x = x + dt * net(x, t_b)
        if return_trajectory:
            traj.append(x.detach().cpu().numpy().copy())
    if return_trajectory:
        return x, np.stack(traj)        # (n_steps+1, N, 2)
    return x


## 9. Train the 1-rectified flow under each scheduler

We sample one shared dataset of independent
$(\mathbf{x}_0, \mathbf{x}_1)$ pairs and train a fresh velocity
network under each scheduler. Each training takes ~30 seconds on
Colab CPU.


In [ ]:
N_TRAIN = 8000

gen = torch.Generator().manual_seed(1)
x0_data = sample_p_src(N_TRAIN, gen).to(device)
x1_data = sample_p_tgt(N_TRAIN, gen).to(device)

nets_k1 = {}
for sch in schedulers:
    print(f"\n=== Training 1-rectified flow with {sch.name} ===")
    torch.manual_seed(42)         # same init for both, isolating the scheduler
    nets_k1[sch.name] = train_velocity(
        x0_data, x1_data, sch, n_iters=3500, label=f"k=1 / {sch.name}")


### Visualize the 1-rectified trajectories

Same fixed test points for both flows — apples to apples.


In [ ]:
def make_test_points(seed=7, n_per_cluster=14):
    np.random.seed(seed)
    pts, ids = [], []
    for i in range(N_CLUSTERS):
        a = i * (2 * np.pi / N_CLUSTERS) + np.pi / N_CLUSTERS
        c = np.array([R_OUTER * np.cos(a), R_OUTER * np.sin(a)])
        pts.append(c + np.random.randn(n_per_cluster, 2) * SIGMA_OUTER)
        ids.extend([i] * n_per_cluster)
    return np.concatenate(pts).astype(np.float32), np.array(ids)

test_x0_np, cluster_ids = make_test_points()
test_x0 = torch.from_numpy(test_x0_np).to(device)

def plot_traj(ax, traj, end, title):
    for j in range(traj.shape[1]):
        c = C_BLUE if cluster_ids[j] % 2 == 0 else C_GREEN
        ax.plot(traj[:, j, 0], traj[:, j, 1], color=c, alpha=0.5, lw=0.95,
                solid_capstyle="round", zorder=2)
    ax.scatter(test_x0_np[:, 0], test_x0_np[:, 1], s=22, c=C_SRC,
               edgecolors="white", linewidths=0.6, zorder=10)
    ax.scatter(end[:, 0], end[:, 1], s=20, c=C_TGT,
               edgecolors="white", linewidths=0.6, zorder=10)
    ax.set_aspect("equal"); ax.set_xlim(-5.8, 5.8); ax.set_ylim(-5.8, 5.8)
    ax.axis("off"); ax.set_title(title, fontsize=14, pad=8)

trajs_k1, ends_k1 = {}, {}
for sch in schedulers:
    end, traj = integrate(nets_k1[sch.name], test_x0,
                          n_steps=200, return_trajectory=True)
    trajs_k1[sch.name] = traj
    ends_k1[sch.name]  = end.detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), facecolor="white")
for ax, sch in zip(axes, schedulers):
    plot_traj(ax, trajs_k1[sch.name], ends_k1[sch.name],
              f"1-rectified · {sch.name}")
plt.subplots_adjust(left=0.01, right=0.99, top=0.92, bottom=0.05, wspace=0.02)
plt.show()


**What we see.**

* **Linear (FM)** produces curved trajectories that land on the
  inner ring. The curvature comes purely from coupling ambiguity:
  each source mode is equidistant from two target modes, so the
  network averages over the choice.
* **VP-Trig (DDPM cosine)** also lands on the inner ring, but its
  trajectories are visibly *more curved* than Linear's. The extra
  curvature is structural — it's the variance-preserving arc baked
  into the conditional path itself. Even with **no** coupling
  ambiguity, VP-Trig would still produce curved trajectories.

> **Key insight.** Linear FM and VP-Trig solve the *same transport
> problem* (both push $p_{\text{src}}$ to $p_{\text{tgt}}$), but
> along **different families of paths**. The shape of the trained
> vector field — and therefore the number of integration steps you
> need at sampling time — depends on the scheduler.


## 10. Reflow under each scheduler

**Recall.** Reflow trains a fresh velocity network on the new
coupling $(\mathbf{z}_0,\,\mathbf{z}_1=\Phi_k(\mathbf{z}_0))$, where
$\Phi_k$ is the trained ODE map of the previous stage. This
removes coupling ambiguity: every $\mathbf{z}_0$ is now
deterministically paired with a single $\mathbf{z}_1$.

**A subtle but important point.** Reflow only straightens
trajectories *if the scheduler's conditional path is already a
straight line*. For Linear FM, the conditional path
$\mathbf{x}_t=(1-t)\mathbf{x}_0+t\mathbf{x}_1$ is a straight
segment, so removing ambiguity produces straight transport. For
VP-Trig, the conditional path is a curved arc — and Reflow leaves
the scheduler unchanged. So as Reflow makes the coupling more
deterministic, the trained flow follows the structural VP-Trig arc
*more* faithfully, and the trajectories actually become **more**
curved.

We'll see this happen in the figure and the straightness table:
Linear's straightness should monotonically approach 1, while
VP-Trig's should *decrease* with successive reflow stages.

**Implementation note.** When we generate the new coupling we use
a coarse Euler integrator (8 steps). The resulting overshoot/
undershoot keeps residual ambiguity in the coupling so that $k=2$
and $k=3$ have measurable progression. With infinite-precision
integration, one round already extracts most of the structure.


In [ ]:
N_STEPS_REFLOW = 8

# Stage k=2 for each scheduler
nets_k2 = {}
for sch in schedulers:
    print(f"\n=== Reflow stage k=2 with {sch.name} ===")
    gen = torch.Generator().manual_seed(100 + hash(sch.name) % 1000)
    z0 = sample_p_src(N_TRAIN, gen).to(device)
    with torch.no_grad():
        z1 = integrate(nets_k1[sch.name], z0, n_steps=N_STEPS_REFLOW)
    torch.manual_seed(43)
    nets_k2[sch.name] = train_velocity(
        z0, z1, sch, n_iters=3500, label=f"k=2 / {sch.name}")

# Stage k=3 for each scheduler
nets_k3 = {}
for sch in schedulers:
    print(f"\n=== Reflow stage k=3 with {sch.name} ===")
    gen = torch.Generator().manual_seed(200 + hash(sch.name) % 1000)
    z0 = sample_p_src(N_TRAIN, gen).to(device)
    with torch.no_grad():
        z1 = integrate(nets_k2[sch.name], z0, n_steps=N_STEPS_REFLOW)
    torch.manual_seed(44)
    nets_k3[sch.name] = train_velocity(
        z0, z1, sch, n_iters=3500, label=f"k=3 / {sch.name}")


## 11. Final 2 × 3 comparison

Rows: schedulers. Columns: rectification level (k=1, 2, 3).


In [ ]:
all_nets = {1: nets_k1, 2: nets_k2, 3: nets_k3}

trajs_grid, ends_grid = {}, {}
for k in (1, 2, 3):
    for sch in schedulers:
        end, traj = integrate(all_nets[k][sch.name], test_x0,
                              n_steps=200, return_trajectory=True)
        trajs_grid[(sch.name, k)] = traj
        ends_grid[(sch.name, k)]  = end.detach().cpu().numpy()

fig, axes = plt.subplots(2, 3, figsize=(14.5, 9.8), facecolor="white")
for r, sch in enumerate(schedulers):
    for c, k in enumerate((1, 2, 3)):
        plot_traj(axes[r, c], trajs_grid[(sch.name, k)],
                  ends_grid[(sch.name, k)],
                  title="" if r != 0 else f"({chr(97+c)}) {k}-rectified flow")
        if c == 0:
            axes[r, c].text(-0.05, 0.5, sch.name,
                            transform=axes[r, c].transAxes,
                            rotation=90, va="center", ha="center",
                            fontsize=14, color=sched_colors[r])

plt.subplots_adjust(left=0.03, right=0.99, top=0.95, bottom=0.02,
                    wspace=0.02, hspace=0.05)
plt.show()


## 12. Quantitative: how straight is "straight"?

Same metric as before: chord length $\|\mathbf{x}(1)-\mathbf{x}(0)\|$
divided by arc length $\int_0^1 \|\dot{\mathbf{x}}(t)\|\,dt$. A
value of $1.0$ means the trajectory is a perfect straight line.


In [ ]:
def straightness(traj):
    chord = np.linalg.norm(traj[-1] - traj[0], axis=-1)
    arc   = np.sum(np.linalg.norm(np.diff(traj, axis=0), axis=-1), 0)
    return float(np.mean(chord / np.maximum(arc, 1e-9)))

print(f"{'Scheduler':<18} {'k=1':>10} {'k=2':>10} {'k=3':>10}")
print("-" * 50)
for sch in schedulers:
    row = [straightness(trajs_grid[(sch.name, k)]) for k in (1, 2, 3)]
    print(f"{sch.name:<18} {row[0]:>10.4f} {row[1]:>10.4f} {row[2]:>10.4f}")


Notice the asymmetry:

* **Linear FM** monotonically approaches $1.0$ — exactly as Reflow
  advertises in the original paper.
* **VP-Trig** *decreases* with each Reflow stage. That's not a bug.
  At $k=1$, ambiguity-induced curvature is mixed with the
  structural arc, partially canceling out — straightness lands
  somewhere in the middle. As Reflow removes the ambiguity, the
  pure structural arc emerges, and the trajectories become more
  tightly curved. The straightness number at $k=3$ is essentially
  the geometric arc-length ratio of the variance-preserving
  path itself.

The lesson: **Reflow is a coupling-deterministicizer, not a path
straightener.** It happens to straighten Linear FM trajectories
only because Linear FM's conditional path is already a straight
line. Choose a curved scheduler and Reflow won't fix that.

## 13. What we learned

1. **The rectified-flow algorithm is scheduler-agnostic.** The
   same code, with one substitution of the conditional path
   $\mathbf{x}_t = \alpha_t \mathbf{x}_0 + \sigma_t \mathbf{x}_1$
   and its derivative, trains a valid flow under any choice with
   $\alpha_0=1, \sigma_0=0, \alpha_1=0, \sigma_1=1$.

2. **Linear and VP-Trig give different conditional paths.** Linear
   is a straight segment ($\alpha_t + \sigma_t = 1$); VP-Trig is a
   curved arc that bulges outward
   ($\alpha_t^2 + \sigma_t^2 = 1$). Both transport
   $p_{\text{src}}$ to $p_{\text{tgt}}$ correctly.

3. **Two sources of curvature in the trained flow.** Coupling
   ambiguity (the network averaging over multiple plausible
   targets) produces one kind of curvature; the structural arc of
   the scheduler produces another. At $k=1$, the trained
   trajectories combine both.

4. **Reflow only removes the first kind.** Reflow rebuilds the
   coupling around the trained ODE map, killing ambiguity-induced
   curvature. It cannot change the scheduler, so the structural
   arc is preserved (and in fact becomes more visible as the
   coupling becomes more deterministic). Linear FM gets straighter
   because its structural arc is *already* a straight line. VP-Trig
   gets *more* visibly curved because its structural arc is a
   variance-preserving arc.

5. **Practical implication.** If you want few-step sampling, choose
   a scheduler whose conditional path is straight (Linear FM is
   the natural choice) and apply Reflow to remove ambiguity.
   Schedulers with curved conditional paths (VP-Trig, VP-Linear,
   most diffusion schedules) will require many integration steps
   no matter how much you reflow — that curvature is structural.

### Things to try

* **Compare 1-step sampling.** Use `n_steps=1` in `integrate` for
  each (scheduler, $k$) cell and measure how close the resulting
  distribution is to $p_{\text{tgt}}$. Linear FM at $k=3$ should
  be excellent; VP-Trig at $k=3$ should be terrible — *worse*
  than VP-Trig at $k=1$.
* **Add a third scheduler.** Try a polynomial schedule like
  $\alpha_t = (1-t)^2$, $\sigma_t = 1-(1-t)^2$. It satisfies the
  endpoints, but the conditional path is yet another curve —
  does Reflow straighten it?
* **Sweep `N_STEPS_REFLOW`** from 4 to 100. For Linear FM, finer
  integration during coupling generation lets one Reflow stage
  absorb most of the curvature. For VP-Trig, no amount of
  integration precision changes the outcome — the structural arc
  is independent of how the coupling was generated.
* **Replace the source** with a Gaussian
  ($\mathbf{x}_0 \sim \mathcal{N}(0, 4I)$). Without coupling
  ambiguity to mask it, the structural arc of VP-Trig becomes
  immediately apparent at $k=1$.
